# Transaction and overdraft evaluation

**Run this notebook first. It evaluates saved data on CPU; it does not train or sample a generator.**

1. Fit the supplied overdraft detector once on real accounts; test on separate real accounts and each saved generation.
2. Compare transaction distributions, overdraft-specific patterns, and account relationships.
3. Inspect uncertainty and tradeoffs before choosing a hyperparameter setting.

Target: **negative balance on the current transaction**, not a future overdraft. No loan prediction is used.

Use the project's `.venv` kernel (`uv sync --locked --group tabular-data --inexact` if dependencies are missing).
The full comparison typically takes about 1–3 minutes on CPU for these saved 20k-row runs.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image
from IPython.display import display as show


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/transaction-evaluation-matplotlib")
MULTI_TABLE = ROOT / "implementations/tabular_data/multi_table"

from implementations.tabular_data.multi_table.evaluation.transaction_evaluation import evaluate_saved_runs

## 1. Choose saved runs

These are the five completed guidance runs, with sampling seed 42. To evaluate future experiments, replace `RUN_DIRS` with the paths returned by the tuning notebook. Each must contain `synthetic_final_with_keys.pkl`; feature-only CSVs cannot support account-level checks.

In [ ]:
DATA_DIR = MULTI_TABLE / "data/berka"
SAVED_SWEEP = MULTI_TABLE / "results/experiments/guidance_strength_first_pass"
RUN_DIRS = [SAVED_SWEEP / f"guidance_{scale:g}_seed42" for scale in [0, 0.5, 1, 2, 4]]
OUTPUT_ROOT = MULTI_TABLE / "results/experiments/transaction_overdraft"
BOOTSTRAP_REPETITIONS = 200
EVALUATION_SEED = 42
THRESHOLD = 0.5  # Fixed diagnostic threshold; not tuned on test/synthetic data.
assert all((p / "synthetic_final_with_keys.pkl").is_file() for p in RUN_DIRS)

## 2. Run the shared evaluation

**One detector, same real split, same threshold for every run.** Inputs are transaction date, amount, transaction type, operation, payment symbol, and bank. Balance creates the label and is excluded from inputs; IDs are also excluded.

The report saves the split, counts, per-row predictions, resolved source fingerprints, per-column and per-category diagnostics, and plots. New evaluation code or changed source files produce a new report directory.

In [ ]:
comparison, REPORT_DIR = evaluate_saved_runs(
    DATA_DIR,
    RUN_DIRS,
    OUTPUT_ROOT,
    bootstrap_repetitions=BOOTSTRAP_REPETITIONS,
    seed=EVALUATION_SEED,
    threshold=THRESHOLD,
)
print(f"Report saved to: {REPORT_DIR}")
show(pd.read_json(REPORT_DIR / "real_split_counts.json"))

## 3. Overdraft transfer

- **Average precision (AP):** how well overdrafts appear near the top of the detector's scores. Report prevalence too: more positives can increase AP without better relationships.
- **Standardized AP:** reweights each dataset to the real test event rate. This controls the class-prior difference only, not task difficulty.
- **ROC-AUC:** a complementary ranking measure; it can look reasonable while precision is poor for rare events.
- **Intervals:** 95% percentile intervals from resampling whole accounts, holding the fitted detector fixed. They do not cover generator-training or generation-seed variability.

The real holdout is independent of detector fitting. It was **not excluded before generator training**, so this is exploratory transfer evaluation, not a claim of downstream generalization or privacy.

In [ ]:
TRANSFER_COLUMNS = [
    "rows",
    "overdrafts",
    "overdraft_prevalence",
    "average_precision",
    "average_precision_low",
    "average_precision_high",
    "standardized_ap",
    "roc_auc",
    "precision",
    "recall",
    "bootstrap_valid",
]
show(comparison[TRANSFER_COLUMNS].round(4))
show(Image(filename=str(REPORT_DIR / "transfer_intervals.png")))
show(Image(filename=str(REPORT_DIR / "precision_recall.png")))

## 4. Transaction quality and account relationships

**Lower distances mean closer to the real reference**, except validity/uniqueness where 1 is desired. Full real training data is the fidelity reference.

Numerical KS covers date, amount, and balance; categorical TVD covers type, operation, symbol, and bank. IDs, including the external counterparty number, are excluded from these measurements. Conditional distances compare overdraft transactions separately, so rare-event failures cannot hide behind the majority.

Account diagnostics include transaction counts (including zero-transaction accounts), observed volume, accounts ever overdrawn, parent/child attribute relationships, valid links, and transaction-before-opening rates. Different date epochs are aligned. Incomplete sampled histories prevent reliable rolling-history and balance-recurrence checks.

In [ ]:
QUALITY_COLUMNS = [
    "transaction_ks",
    "transaction_tvd",
    "overdraft_numerical_ks",
    "overdraft_categorical_tvd",
    "transactions_per_account_ks",
    "account_transaction_correlation_rmse",
    "accounts_overdrawn_rate_gap",
    "transaction_fk_valid_rate",
    "transaction_pk_unique_rate",
    "synthetic_transaction_before_account_rate",
    "max_invalid_rate",
    "max_unknown_category_rate",
    "undefined_account_pairs",
]
show(comparison.loc[comparison.index != "real_test", QUALITY_COLUMNS].round(4))
show(Image(filename=str(REPORT_DIR / "quality_comparison.png")))

## 5. Inspect a run

Category rates include row and event counts. Small categories can have large errors by chance; inspect support before interpreting them. Category labels come from the saved preprocessing codebook. Monthly rates and all correlation pairs are also saved as CSVs.

In [ ]:
INSPECT_RUN = RUN_DIRS[1].name
DETAIL_DIR = REPORT_DIR / INSPECT_RUN
show(pd.read_csv(DETAIL_DIR / "distributions.csv").round(4))
show(pd.read_csv(DETAIL_DIR / "conditional_distributions.csv").query("overdraft == 1").round(4))
groups = pd.read_csv(DETAIL_DIR / "overdraft_by_category.csv")
show(groups.sort_values("rate_error", ascending=False).head(12).round(4))

## Conclusion — saved guidance runs, seed 42

1. **No confirmed winner for overdraft quality.** Synthetic AP is 0.014–0.038 versus 0.170 on real held-out accounts. Only **8 real test overdrafts** make the baseline uncertain (approximately 0.002–0.557 bootstrap AP interval). Guidance 0 has the highest synthetic AP; this alone is insufficient for selection.
2. **Guidance 0–0.5 deserves further study.** Guidance 0.5 has the closest overall numerical and categorical transaction distributions; guidance 4 worsens both. Overdraft-only numerical differences remain large (KS roughly 0.43–0.53), even when overall distributions look reasonable.
3. **Account chronology is a major unresolved failure:** about **21–25%** of synthetic transactions occur before their account opens; real data has zero. Foreign-key validity is perfect but does not ensure realistic relationships.

**Next research experiment:** retain 0, 0.5, and 1 as candidates; establish a fresh baseline with aligned diffusion/classifier preprocessing splits, then tune clustering/parent weight. Confirm promising settings with more generation seeds and inspect rare-event and chronology diagnostics together. No new GPU experiments were run for this report.

These conclusions describe the listed five saved runs. Reassess them after changing `RUN_DIRS`; do not automatically select the highest AP or hide tradeoffs in one combined score.